In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl
import seaborn as sns

from climate_attitudes import configure_mpl
from climate_attitudes.dataset import Dataset
from climate_attitudes.settings import Config
from climate_attitudes.visualisation import QUALITATIVE_SCHEME

FONT_PATH = Path("../fonts")
configure_mpl(FONT_PATH)

plt.rc("figure", dpi=150)

In [ ]:
config = Config(_env_file="../.env")
dataset = Dataset.load(config, with_imputation=False)
resp = dataset.response.collect()

In [ ]:
time_deltas = (
    resp.select("participant_id", "wave", "start_date")
    .sort(by=("participant_id", "wave"))
    .with_columns(
        pl.col("wave").diff().over("participant_id").alias("wave_diff"),
        pl.col("wave").min().over("participant_id").alias("first_wave"),
    )
    .filter((pl.col("wave_diff") == 1) | (pl.col("wave") == pl.col("first_wave")))
    .with_columns(
        pl.col("start_date").diff().over("participant_id").alias("time_delta")
    )
    .filter(pl.col("time_delta").is_not_null())
    .with_columns(pl.col("time_delta").dt.total_days())
    .drop("start_date", "first_wave", "wave_diff")
)

In [ ]:
from climate_attitudes.cli.common import normalise_raw_response_schema
from climate_attitudes.settings import RawDataFile
from climate_attitudes.visualisation import configure_mpl

w1_to_5 = normalise_raw_response_schema(
    RawDataFile.Waves1to5Responses.scan(config)
).collect()
w6 = normalise_raw_response_schema(RawDataFile.Wave6Responses.scan(config)).collect()

df = pl.concat(
    [
        w1_to_5.select("participant_id", "wave", "start_date"),  # ty: ignore
        w6.select("participant_id", "wave", "start_date"),  # ty: ignore
    ]
).filter(pl.col("participant_id").is_not_null())

intervals = pl.Enum([f"{i - 1}--{i}" for i in range(2, 7)])

time_deltas = (
    df.sort(by=("participant_id", "wave"))
    .with_columns(
        pl.col("wave").diff().over("participant_id").alias("wave_diff"),
        pl.col("wave").min().over("participant_id").alias("first_wave"),
    )
    .filter((pl.col("wave_diff") == 1) | (pl.col("wave") == pl.col("first_wave")))
    .with_columns(
        pl.col("start_date").diff().over("participant_id").alias("time_delta")
    )
    .filter(pl.col("time_delta").is_not_null())
    .with_columns(pl.col("time_delta").dt.total_days())
    .drop("start_date", "first_wave", "wave_diff")
)
time_deltas = time_deltas.with_columns(
    (pl.col("wave") - 2).cast(intervals).cast(int)
).rename({"wave": "Interval"})
# time_deltas = time_deltas.rename({"wave": "Interval"})

In [ ]:
time_deltas = (
    df.join(df, on="participant_id")
    .filter(pl.col("wave_right") == (pl.col("wave") - 1))
    .with_columns(
        time_delta=(pl.col("start_date") - pl.col("start_date_right")).dt.total_days()
    )
    .drop("start_date", "wave_right", "start_date_right")
    .rename({"wave": "Interval"})
    .sort(by=("Interval"))
    .with_columns((pl.col("Interval") - 2).cast(intervals).cast(pl.String))
)

In [ ]:
time_deltas

In [ ]:
time_deltas

In [ ]:
fig, ax = plt.subplots(figsize=(3.5, 2), constrained_layout=True)

sns.kdeplot(
    time_deltas,
    x="time_delta",
    hue="Interval",
    fill=True,
    ax=ax,
    palette=QUALITATIVE_SCHEME.colors[
        : time_deltas.select(pl.col("Interval").n_unique()).item()
    ],
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.set_xlabel(r"$\Delta t$ (days)");

In [ ]:
time_deltas.filter(pl.col("time_delta") == pl.col("time_delta").max())

In [ ]:
(
    df.join(df, on="participant_id")
    .filter(pl.col("wave_right") == (pl.col("wave") - 1))
    .with_columns(
        time_delta=(pl.col("start_date") - pl.col("start_date_right")).dt.total_days()
    )
    .drop("start_date", "wave_right", "start_date_right")
    .sort(by=("participant_id", "wave"))
)

In [ ]:
(
    # df
    df.filter(participant_id=1502973531)
    .sort(by=("participant_id", "wave"))
    .with_columns(
        pl.col("wave").diff().over("participant_id").alias("wave_diff"),
        pl.col("wave").min().over("participant_id").alias("first_wave"),
    )
    # .filter(
    #     (pl.col("wave_diff") == 1) | (pl.col("wave") == pl.col("first_wave"))
    # )
    # .with_columns(
    #     pl.col("start_date").diff().over("participant_id").alias("time_delta")
    # )
    # .filter(pl.col("time_delta").is_not_null())
    # .with_columns(pl.col("time_delta").dt.total_days())
    # .drop("start_date", "first_wave", "wave_diff")
    # .filter(participant_id=2010178239)
)

In [ ]:
df.filter(participant_id=1502973531)